# BG7TBL WB-SG2-20G Signal Generator — Control Notebook

**Instrument:** BG7TBL WB-SG2-20G Wideband Signal Generator  
**CH1:** 1 Hz – 250 MHz (BNC, 3.3 Vpp)  
**CH2:** 10 MHz – 20 GHz (SMA, 50 Ω)  
**Interface:** USB serial via FTDI — uses `pyserial`, NOT pyvisa  
**Protocol:** Proprietary `$X...*` text commands (community reverse-engineered, verified on this unit)

---

> ## ✅ Verified Command Set
> | Function | Command | Notes |
> |---|---|---|
> | Query status | `$A*` | Returns full status string (parsed below) |
> | CH1 frequency | `$F1` + 10 digits Hz | e.g. `$F1010000000*` ; DDS quantizes, read back actual |
> | CH2 frequency | `$F2` + 11 digits Hz | e.g. `$F200001000000*` = 1 GHz |
> | Power level | `$P` + 2 digits | Range 00–63; write-only, echoes `SET PWR NN POK` |
> | CH1 sweep | `$W3<start><stop><point>*` | 10-digit fields |
> | CH2 sweep | `$W4<start><stop><point>*` | 11-digit fields |
> | Save EEPROM | `$S*` | Also acts as ENT key |
> | LCD contrast | `$C` + 2 digits | Range 00–63 ONLY |
> | Keys | `$U` `$D` `$L` `$R` `$N` `$S` | Up/Down/Left/Right/Mode/Enter |
>
> **Front-panel only (no direct serial command — use key emulation):**
> Output ON/OFF · Modulation ON/OFF · Reference INT/EXT.
> The device parser matches the first 1–2 chars after `$` and treats the rest
> as a numeric argument, so multi-letter commands like `$MODON` collide with
> single-letter key commands and do nothing useful.

---

> ## ⚠️ Cautions
> - **`$C` contrast: never send ≥ 64** — the LCD goes white. Recovery: `$C45*`.
> - **Never send `$B` (baud change)** — it can drop the connection until power-cycle.
> - **Never send `$BP...`** — observed to hang the FTDI link (power-cycle to recover).
> - Frequency fields must be the exact digit count, all numeric. Wrong length
>   makes the parser read garbage (e.g. `$F1OFF` set CH1 to a random frequency).
> - CH1 uses a DDS with finite resolution — always read back the actual frequency.

---
**How to use:**
1. Install pyserial: `pip install pyserial`
2. Set `PORT` in the config block (Cell 2).
3. Run imports/connection (Cell 3) and function definitions (Cell 4).
4. Use the example cells or call functions directly.

## ⚙️ CONFIGURATION BLOCK

In [ ]:
# ============================================================
#  SERIAL PORT
#  Windows: 'COM9'   Linux: '/dev/ttyUSB0'   Mac: '/dev/tty.usbserial-XXXX'
# ============================================================
PORT      = 'COM9'
BAUDRATE  = 9600       # Device default — do not change
TIMEOUT_S = 1.0        # Serial read timeout
CMD_DELAY = 0.3        # Default seconds to wait after a command before reading

# ============================================================
#  DEVICE LIMITS
# ============================================================
CH1_MIN_HZ = 1
CH1_MAX_HZ = 250_000_000          # 250 MHz
CH2_MIN_HZ = 10_000_000           # 10 MHz
CH2_MAX_HZ = 20_000_000_000       # 20 GHz

CH1_DIGITS = 10                   # CH1 frequency field width
CH2_DIGITS = 11                   # CH2 frequency field width

POWER_MIN  = 0                    # Power level range 00–63
POWER_MAX  = 63

# ============================================================
#  DEFAULTS
# ============================================================
DEFAULT_CH1_HZ = 10_000_000       # 10 MHz
DEFAULT_CH2_HZ = 1_000_000_000    # 1 GHz
DEFAULT_POWER  = 63               # Max power level

print("WB-SG2-20G configuration loaded.")

## 📦 Imports & Connection

In [ ]:
import serial
import serial.tools.list_ports
import time
import re

# List available ports for reference
print("Available serial ports:")
for p in serial.tools.list_ports.comports():
    print(f"  {p.device:<14} {p.description}")

# Open the connection
sg = serial.Serial(
    port     = PORT,
    baudrate = BAUDRATE,
    bytesize = serial.EIGHTBITS,
    parity   = serial.PARITY_NONE,
    stopbits = serial.STOPBITS_ONE,
    timeout  = TIMEOUT_S
)
sg.reset_input_buffer()
sg.reset_output_buffer()
time.sleep(0.2)
print(f"\nOpened {sg.name} at {sg.baudrate} baud")

## 🔧 Function Definitions

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  LOW-LEVEL I/O
# ══════════════════════════════════════════════════════════════════════════════

def _send(cmd, delay=CMD_DELAY):
    """
    Send a raw command and return the decoded response string.
    Automatically wraps the command in $...* if not already.

    Parameters
    ----------
    cmd   : Command string, with or without $ and * delimiters
    delay : Seconds to wait after writing before reading

    Returns
    -------
    str : decoded response (stripped), or '' if no response
    """
    if not cmd.startswith('$'):
        cmd = '$' + cmd
    if not cmd.endswith('*'):
        cmd = cmd + '*'
    sg.reset_input_buffer()
    sg.write(cmd.encode('ascii'))
    time.sleep(delay)
    raw = sg.read(sg.in_waiting or 256)
    return raw.decode('ascii', errors='replace').strip()


# ══════════════════════════════════════════════════════════════════════════════
#  STATUS QUERY & PARSER
# ══════════════════════════════════════════════════════════════════════════════

def sg_status_raw():
    """Return the raw $A* status string."""
    return _send('$A*')


def sg_status():
    """
    Query and parse the device status into a dict.

    The status string looks like:
    SYS STATE:CH1 0099999994Hz,OUT ON,CH2 10000000000Hz,OUT ON,
    MOD OFF,PLL LOCK,REF EXT,BAUARD 0000009600 BPS,AOK

    Returns
    -------
    dict with keys:
        ch1_hz, ch1_out, ch2_hz, ch2_out, mod, pll, ref, baud, raw
    """
    raw = sg_status_raw()
    s = {'raw': raw}
    try:
        # CH1 frequency and output
        m = re.search(r'CH1 (\d+)Hz,OUT (ON|OFF)', raw)
        if m:
            s['ch1_hz']  = int(m.group(1))
            s['ch1_out'] = m.group(2)
        # CH2 frequency and output
        m = re.search(r'CH2 (\d+)\*?Hz,OUT (ON|OFF)', raw)
        if m:
            s['ch2_hz']  = int(m.group(1))
            s['ch2_out'] = m.group(2)
        # Modulation, PLL, reference
        m = re.search(r'MOD (ON|OFF)', raw)
        if m: s['mod'] = m.group(1)
        m = re.search(r'PLL (LOCK|UNLOCK)', raw)
        if m: s['pll'] = m.group(1)
        m = re.search(r'REF (EXT|INT)', raw)
        if m: s['ref'] = m.group(1)
        m = re.search(r'BAUARD (\d+) BPS', raw)
        if m: s['baud'] = int(m.group(1))
    except Exception as e:
        print(f"Parse warning: {e}")
    return s


def sg_print_status():
    """Query and pretty-print the current device status."""
    s = sg_status()
    if 'ch1_hz' not in s:
        print("No valid status — check connection.")
        print(f"Raw: {s['raw']!r}")
        return s
    print("── WB-SG2-20G Status ──────────────────────────────────")
    print(f"  CH1 : {s['ch1_hz']:>13,} Hz   OUT {s.get('ch1_out','?')}")
    print(f"  CH2 : {s['ch2_hz']:>13,} Hz   OUT {s.get('ch2_out','?')}")
    print(f"  MOD : {s.get('mod','?')}")
    print(f"  PLL : {s.get('pll','?')}")
    print(f"  REF : {s.get('ref','?')}")
    print(f"  BAUD: {s.get('baud','?')}")
    return s


# ══════════════════════════════════════════════════════════════════════════════
#  FREQUENCY CONTROL
# ══════════════════════════════════════════════════════════════════════════════

def set_ch1_frequency(freq_hz=DEFAULT_CH1_HZ):
    """
    Set CH1 output frequency (1 Hz – 250 MHz).
    CH1 uses a DDS with finite resolution; the actual tuned frequency is
    read back from the device and returned (may differ slightly from request).

    Parameters
    ----------
    freq_hz : Target frequency in Hz

    Returns
    -------
    actual_hz : int — frequency confirmed by the device, or None on error
    """
    if not (CH1_MIN_HZ <= freq_hz <= CH1_MAX_HZ):
        raise ValueError(f"CH1 frequency {freq_hz} Hz out of range "
                         f"({CH1_MIN_HZ}–{CH1_MAX_HZ} Hz)")
    cmd  = f'$F1{int(freq_hz):0{CH1_DIGITS}d}*'
    resp = _send(cmd)
    # Response like 'CH1 0099999994Hz FOK'
    m = re.search(r'CH1 (\d+)Hz', resp)
    if m:
        actual = int(m.group(1))
        print(f"CH1: requested {freq_hz:,} Hz → actual {actual:,} Hz")
        return actual
    print(f"CH1 set response: {resp!r}")
    return None


def set_ch2_frequency(freq_hz=DEFAULT_CH2_HZ):
    """
    Set CH2 output frequency (10 MHz – 20 GHz).

    Parameters
    ----------
    freq_hz : Target frequency in Hz

    Returns
    -------
    actual_hz : int — frequency confirmed by the device, or None on error
    """
    if not (CH2_MIN_HZ <= freq_hz <= CH2_MAX_HZ):
        raise ValueError(f"CH2 frequency {freq_hz} Hz out of range "
                         f"({CH2_MIN_HZ}–{CH2_MAX_HZ} Hz)")
    cmd  = f'$F2{int(freq_hz):0{CH2_DIGITS}d}*'
    resp = _send(cmd)
    # Response like 'CH2 01000000000Hz FOK'
    m = re.search(r'CH2 (\d+)Hz', resp)
    if m:
        actual = int(m.group(1))
        print(f"CH2: requested {freq_hz:,} Hz → actual {actual:,} Hz")
        return actual
    print(f"CH2 set response: {resp!r}")
    return None


# ══════════════════════════════════════════════════════════════════════════════
#  POWER LEVEL
#  Range 00–63. Write-only — not reported in status. The echo is the only
#  confirmation. (The exact dBm mapping is not documented by the vendor.)
# ══════════════════════════════════════════════════════════════════════════════

def set_power(level=DEFAULT_POWER):
    """
    Set the output power level (0–63).

    Parameters
    ----------
    level : Power level integer 0–63 (higher = more output)

    Returns
    -------
    level : int — the level set, or None on error
    """
    if not (POWER_MIN <= level <= POWER_MAX):
        raise ValueError(f"Power level {level} out of range "
                         f"({POWER_MIN}–{POWER_MAX})")
    resp = _send(f'$P{int(level):02d}*')
    m = re.search(r'SET PWR (\d+) POK', resp)
    if m:
        print(f"Power level set to {int(m.group(1))}")
        return int(m.group(1))
    print(f"Power set response: {resp!r}")
    return None


# ══════════════════════════════════════════════════════════════════════════════
#  SWEEP
#  $W3 = CH1 sweep (10-digit fields), $W4 = CH2 sweep (11-digit fields).
#  Format: $W3<start><stop><point>*  where point = step count or step size.
# ══════════════════════════════════════════════════════════════════════════════

def set_ch1_sweep(start_hz, stop_hz, point):
    """
    Configure a CH1 frequency sweep.

    Parameters
    ----------
    start_hz : Sweep start frequency Hz
    stop_hz  : Sweep stop frequency Hz
    point    : Sweep point/step value (10-digit field)

    Returns
    -------
    str : device response
    """
    cmd = (f'$W3{int(start_hz):0{CH1_DIGITS}d}'
           f'{int(stop_hz):0{CH1_DIGITS}d}'
           f'{int(point):0{CH1_DIGITS}d}*')
    resp = _send(cmd, delay=0.5)
    print(f"CH1 sweep: {resp}")
    return resp


def set_ch2_sweep(start_hz, stop_hz, point):
    """
    Configure a CH2 frequency sweep.

    Parameters
    ----------
    start_hz : Sweep start frequency Hz
    stop_hz  : Sweep stop frequency Hz
    point    : Sweep point/step value (11-digit field)

    Returns
    -------
    str : device response
    """
    cmd = (f'$W4{int(start_hz):0{CH2_DIGITS}d}'
           f'{int(stop_hz):0{CH2_DIGITS}d}'
           f'{int(point):0{CH2_DIGITS}d}*')
    resp = _send(cmd, delay=0.5)
    print(f"CH2 sweep: {resp}")
    return resp


def exit_sweep():
    """Exit sweep mode by emulating the Mode key (returns to point-frequency)."""
    resp = _send('$N*')
    print(f"Exit sweep: {resp}")
    return resp


# ══════════════════════════════════════════════════════════════════════════════
#  FRONT-PANEL KEY EMULATION
#  Output ON/OFF, modulation, and reference selection have no direct serial
#  command — navigate the on-screen menu with these keys to change them.
# ══════════════════════════════════════════════════════════════════════════════

def key_up():
    """Press the Up key."""
    return _send('$U*')

def key_down():
    """Press the Down key."""
    return _send('$D*')

def key_left():
    """Press the Left key."""
    return _send('$L*')

def key_right():
    """Press the Right key."""
    return _send('$R*')

def key_mode():
    """Press the Mode key (cycles operating modes / exits sweep)."""
    return _send('$N*')

def key_enter():
    """Press the Enter key (also triggers EEPROM save)."""
    return _send('$S*')


# ══════════════════════════════════════════════════════════════════════════════
#  SYSTEM / UTILITY
# ══════════════════════════════════════════════════════════════════════════════

def save_eeprom():
    """Save current settings to EEPROM (emulates Enter key)."""
    resp = _send('$S*')
    print(f"Save: {resp}")
    return resp


def set_contrast(level=45):
    """
    Set the LCD contrast.

    Parameters
    ----------
    level : Contrast 0–63. ⚠️ Values ≥ 64 turn the LCD white — never exceed 63.
    """
    if not (0 <= level <= 63):
        raise ValueError("Contrast must be 0–63 (≥64 turns the LCD white).")
    resp = _send(f'$C{int(level):02d}*')
    print(f"Contrast: {resp}")
    return resp


print("WB-SG2-20G functions defined — ready.")

---
## 🧪 Examples

In [ ]:
# ── Example 1: Read and display current status ───────────────────────────────
sg_print_status()

In [ ]:
# ── Example 2: Set CH1 to 10 MHz (note DDS quantization in readback) ─────────
actual = set_ch1_frequency(10_000_000)

In [ ]:
# ── Example 3: Set CH2 to 2.4 GHz ────────────────────────────────────────────
actual = set_ch2_frequency(2_400_000_000)

In [ ]:
# ── Example 4: Set power level ───────────────────────────────────────────────
set_power(63)    # Maximum (range 0–63)

In [ ]:
# ── Example 5: CH2 sweep 1–2 GHz ─────────────────────────────────────────────
set_ch2_sweep(start_hz=1_000_000_000, stop_hz=2_000_000_000, point=100_000_000)
# When done, return to point-frequency mode:
# exit_sweep()

In [ ]:
# ── Example 6: Toggle output via front-panel keys ────────────────────────────
# Output ON/OFF has no direct serial command. Navigate the menu with keys.
# The exact sequence depends on the current menu position — watch the LCD.
# Example: press Mode to reach the output field, then Up/Down to toggle.
# key_mode()
# key_up()
print("Output, modulation, and reference are front-panel-only — use key_* functions.")

In [ ]:
# ── Example 7: Save current settings to EEPROM ───────────────────────────────
# save_eeprom()

---
## 🔌 Cleanup

In [ ]:
sg.close()
print("Serial port closed.")